# Notebook 1: Prepping the data

Turning raw data (GBIF mammal records, UKCEH land cover, OS roads and ONS boundaries) into two clean tables:

- Habitat variables for every 100m cell in Greater London
- Which cells  have mammal and hedgehog records

## Settings

Loading the libraries and setting the main choices in one place. 
Everything is converted to British National Grid (BNG) so distances are in metres, and records that are less precise than 250m are dropped because they can't be placed in a single 100m cell.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio as rio
from rasterio.windows import Window
from rasterio.features import rasterize
from rasterio.transform import from_origin, from_origin, rowcol, xy
from scipy import ndimage

RAW = Path("../data/raw")
OUT = Path("../data/processed")
FIGS = Path("../output/figures")

BNG = "EPSG:27700"  # GBIF uses degrees, but we want metres for 100m cells and 500m radii
CELL = 100  # 100m = 1 hectare, enough detail such as parks
RADIUS = 500  # 500m is a rough area of a hedgeshogs home range
MAX_UNCERTAINTY = 250  # maximum uncertainty in metres for GBIF records to be included
HEDGEHOG = "Erinaceus europaeus"  # scientific name for hedgehogs

## London boundary

Loading the ONS council boundaries, keeping 33 areas with codes starting with E09 (32 London boroughs and City of London).
Converting them to BNG so they're in metres (m).
Saved for later mapping and borough totals, then merged into a single outline which defines the study area.

In [3]:
lad = gpd.read_file(RAW / "lad_boundaries.geojson")
code_col = next(c for c in lad.columns if c.upper().startswith("LAD") and c.upper().endswith("CD"))
name_col = next(c for c in lad.columns if c.upper().startswith("LAD") and c.upper().endswith("NM"))

boroughs = lad[lad[code_col].str.startswith("E09")].to_crs(BNG)
print(len(boroughs), "London areas")   # should print 33
boroughs = boroughs[[code_col, name_col, "geometry"]].rename(columns={code_col: "code", name_col: "borough"})
boroughs.to_file(OUT / "london_boroughs.gpkg", driver="GPKG")
london = boroughs.dissolve()

33 London areas
